# Bond-probability diagnostic

Purpose: check whether the model's predicted edge probability `p₁ = bonds[..., 1]`
carries a geometry-aware signal, even though argmax never picks it.

Trees have a strong distance prior: real edges are almost always between
**close** pairs. If the model learned anything useful, `p₁` on a given pair
should be **decreasing in pairwise L2 distance**. If `p₁` looks uniform with
respect to distance → the model has not learned bond structure and needs
retraining. If `p₁` clearly tracks distance → argmax is the bottleneck and
threshold / spanning-tree decoding will unlock edges without retraining.

Input: `<save_file>.raw.pt` produced by `sample_neurons.py --save_raw`.

Dependencies: `plotly`, `ipywidgets`, `matplotlib` (all used in the visualiser notebook already).

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import IntSlider, IntText, FloatSlider, HBox, VBox, jslink
from IPython.display import display

import semlaflow.scriptutil as util
from semlaflow.data.swc import NEURON_EDGE_CLASS_INDEX

In [ ]:
RAW_PATH = Path("../neuron_samples_atomtypeloss.raw.pt")   # edit if different
COORDS_STD = util.NEURON_COORDS_STD_DEV

assert RAW_PATH.exists(), f"Missing {RAW_PATH.resolve()}. Re-run sample_neurons with --save_raw."

raw_batches = torch.load(RAW_PATH, map_location="cpu", weights_only=False)
print(f"Loaded {len(raw_batches)} batches.")
print(f"Batch 0 keys: {list(raw_batches[0].keys())}")
for k, v in raw_batches[0].items():
    if torch.is_tensor(v):
        print(f"  {k}: {tuple(v.shape)} {v.dtype}")

## Flatten batches into a per-sample list

Each entry holds the tensors for one sample: coords `[N,3]` (physical units), `p₁ [N,N]`, `p₀ [N,N]`, and `argmax_class [N,N]` (so we can see whether the model was ever *close* to picking class 1).

In [ ]:
def flatten_batches(raw_batches):
    samples = []
    for batch in raw_batches:
        coords = batch["coords"]     # [B, Nmax, 3]
        bonds = batch["bonds"]       # [B, Nmax, Nmax, n_bond_types]
        mask = batch["mask"].bool()  # [B, Nmax]
        B = coords.size(0)
        for b in range(B):
            n = int(mask[b].sum().item())
            if n == 0:
                continue
            c = coords[b, :n].float() * COORDS_STD
            bd = bonds[b, :n, :n].float()      # [n, n, n_bond_types]
            samples.append({
                "coords": c.numpy(),
                "bond_dists": bd.numpy(),
                "argmax": bd.argmax(dim=-1).numpy(),
            })
    return samples

samples = flatten_batches(raw_batches)
n_bond_types = samples[0]["bond_dists"].shape[-1]
print(f"{len(samples)} samples, n_bond_types = {n_bond_types}")
print(f"Sample 0: n = {samples[0]['coords'].shape[0]}")

## Sanity check: is the model ever picking class 1 at all?

For each sample, tabulate how often each bond class wins the argmax on real (off-diagonal) pairs. If class 1 literally never wins anywhere, confirms the argmax-bottleneck narrative.

In [ ]:
def class_counts(s):
    n = s["coords"].shape[0]
    iu = np.triu_indices(n, k=1)
    a = s["argmax"][iu]
    counts = np.bincount(a, minlength=n_bond_types)
    return counts

all_counts = np.stack([class_counts(s) for s in samples], axis=0)  # [n_samples, n_bond_types]
print("Off-diagonal argmax class counts (summed over all samples):")
totals = all_counts.sum(axis=0)
total_pairs = totals.sum()
for c, tot in enumerate(totals):
    print(f"  class {c}: {tot:>8d}  ({100 * tot / max(total_pairs, 1):5.2f}%)")

## The key test: does `p₁` drop with pairwise distance?

Aggregated across ALL samples: bucket pairs by pairwise distance, plot mean `p₁` per bucket. A downward trend means geometry-aware learning. A flat line means no learning.

In [ ]:
def collect_pair_stats(samples):
    dists, p1s, p0s = [], [], []
    for s in samples:
        c = s["coords"]
        bd = s["bond_dists"]
        n = c.shape[0]
        iu = np.triu_indices(n, k=1)
        pair_d = np.linalg.norm(c[iu[0]] - c[iu[1]], axis=-1)
        p1 = bd[iu[0], iu[1], NEURON_EDGE_CLASS_INDEX]
        p0 = bd[iu[0], iu[1], 0]
        dists.append(pair_d)
        p1s.append(p1)
        p0s.append(p0)
    return np.concatenate(dists), np.concatenate(p1s), np.concatenate(p0s)

D, P1, P0 = collect_pair_stats(samples)
print(f"{D.shape[0]} total pairs across {len(samples)} samples")
print(f"  distance: min {D.min():.2f}, median {np.median(D):.2f}, max {D.max():.2f} (physical units)")
print(f"  p₁: min {P1.min():.4f}, median {np.median(P1):.4f}, max {P1.max():.4f}")
print(f"  p₀: min {P0.min():.4f}, median {np.median(P0):.4f}, max {P0.max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(P1, bins=60)
axes[0].set_title("Distribution of p₁ over ALL pairs")
axes[0].set_xlabel("p₁")
axes[0].set_yscale("log")

axes[1].scatter(D, P1, s=2, alpha=0.2)
axes[1].set_title("p₁ vs pairwise distance")
axes[1].set_xlabel("distance (physical)")
axes[1].set_ylabel("p₁")

# Bucketed mean / stderr
nb = 20
edges = np.quantile(D, np.linspace(0, 1, nb + 1))
centers, means, stderrs = [], [], []
for i in range(nb):
    sel = (D >= edges[i]) & (D < edges[i + 1]) if i < nb - 1 else (D >= edges[i]) & (D <= edges[i + 1])
    if sel.sum() < 5:
        continue
    centers.append(0.5 * (edges[i] + edges[i + 1]))
    means.append(P1[sel].mean())
    stderrs.append(P1[sel].std(ddof=1) / np.sqrt(sel.sum()))
centers, means, stderrs = np.array(centers), np.array(means), np.array(stderrs)
axes[2].errorbar(centers, means, yerr=stderrs, marker="o")
axes[2].set_title("Mean p₁ per distance bucket (quantile-binned)")
axes[2].set_xlabel("distance (physical)")
axes[2].set_ylabel("E[p₁ | distance bucket]")
axes[2].axhline(P1.mean(), ls="--", color="grey", label=f"global mean {P1.mean():.4f}")
axes[2].legend()
plt.tight_layout()
plt.show()

# Spearman correlation (rank-based; robust to outliers)
from scipy.stats import spearmanr
rho, pval = spearmanr(D, P1)
print(f"Spearman(distance, p₁) = {rho:+.4f}  (p = {pval:.2e})")
print("  Expect negative rho if model learned 'close pairs → edges'.")

## Per-sample view: top-K pairs by p₁

If the above shows even a weak negative correlation, this cell lets you scrub through samples and see the top-K most-confident predicted edges overlaid on the 3D point cloud. Visual tree-plausibility is the fastest sanity check.

In [ ]:
def topk_edges(sample, k):
    n = sample["coords"].shape[0]
    iu = np.triu_indices(n, k=1)
    p1 = sample["bond_dists"][iu[0], iu[1], NEURON_EDGE_CLASS_INDEX]
    k = min(k, p1.shape[0])
    idx = np.argpartition(-p1, k - 1)[:k]
    return iu[0][idx], iu[1][idx], p1[idx]


def build_traces(sample, k):
    coords = sample["coords"]
    a, b, p1 = topk_edges(sample, k)
    node_trace = go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers", marker=dict(size=3, color="#1f77b4"), name="nodes",
    )
    xs, ys, zs, cs = [], [], [], []
    # color-encode edges by p₁ via a simple single-trace Scatter3d with colorscale on a marker;
    # Scatter3d-lines doesn't support per-segment color, so we use thickness via widths=2 and hover text.
    hovers = []
    for i in range(len(a)):
        xs.extend([coords[a[i], 0], coords[b[i], 0], None])
        ys.extend([coords[a[i], 1], coords[b[i], 1], None])
        zs.extend([coords[a[i], 2], coords[b[i], 2], None])
        hovers.extend([f"{a[i]}-{b[i]} p₁={p1[i]:.3f}", f"{a[i]}-{b[i]} p₁={p1[i]:.3f}", ""])
    edge_trace = go.Scatter3d(
        x=xs, y=ys, z=zs, mode="lines",
        line=dict(color="#444", width=3),
        name=f"top-{len(a)} by p₁", hoverinfo="text", text=hovers,
    )
    return [edge_trace, node_trace], p1.min(), p1.max()


viewer = go.FigureWidget(
    layout=dict(
        scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        height=650, margin=dict(l=0, r=0, t=40, b=0), showlegend=False,
    )
)

sample_slider = IntSlider(min=0, max=len(samples) - 1, value=0, description="Sample", continuous_update=False)
sample_box = IntText(value=0, description="Idx")
jslink((sample_slider, "value"), (sample_box, "value"))


def topk_default(n):
    return max(1, n - 1)


k_slider = IntSlider(
    min=1, max=max(s["coords"].shape[0] * (s["coords"].shape[0] - 1) // 2 for s in samples),
    value=topk_default(samples[0]["coords"].shape[0]),
    description="top-K", continuous_update=False,
)


def _render(_=None):
    idx = int(sample_slider.value)
    s = samples[idx]
    n = s["coords"].shape[0]
    # cap K to available off-diagonal pairs
    k = min(int(k_slider.value), n * (n - 1) // 2)
    traces, pmin, pmax = build_traces(s, k)
    with viewer.batch_update():
        viewer.data = ()
        for t in traces:
            viewer.add_trace(t)
        viewer.layout.title = (
            f"Sample {idx} — N={n}, top {k} pairs by p₁ — "
            f"p₁ range in shown edges: [{pmin:.3f}, {pmax:.3f}]"
        )


sample_slider.observe(_render, names="value")
k_slider.observe(_render, names="value")
_render()

display(VBox([HBox([sample_slider, sample_box, k_slider]), viewer]))

## Interpretation guide

- **Spearman ρ(distance, p₁) more negative than −0.1 AND the bucketed plot trends downward** → the model has learned a real distance-aware signal. Threshold / spanning-tree decoding will extract usable trees from this checkpoint. No retrain needed.
- **ρ near zero, flat bucketed plot** → the model has not learned bond structure at all. Retrain with explicit positive class weight in bond CE (Fix A in the previous analysis) and on the full corpus.
- **p₁ max across all pairs is also a tell**: if the global max is < 0.1, the bond head's dynamic range is crushed → loss-level fix needed. If it's > 0.3 somewhere but always < p₀ → decoding fix alone is likely enough.